# 반도체 산업 해자 데이터 추출 파이프라인 (WRDS)

GICS 453010(반도체) 전 종목의 분기 재무 + 수정주가를 WRDS에서 추출해 `Moat_Analysis.csv`로 저장한다.

> **실행에는 WRDS 학술 계정이 필요하다.** 계정 없이 분석을 재현하려면 커밋된 CSV를 사용하는
> [`INTC_vs_NVDA_비교분석.ipynb`](INTC_vs_NVDA_비교분석.ipynb)를 실행하면 된다.


In [ ]:
import os
import wrds
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

db = wrds.Connection(
    wrds_username=os.getenv("WRDS_USERNAME")
)

print("WRDS connection successful")

## 해자(moat) 지표 정의

**Gross Margin (매출총이익률)** : (revtq − cogsq) / revtq
해자가 있다면 원재료 값이 올라도 판가를 올릴 수 있으므로 이 수치가 방어되거나 상승한다.

**Marketing Efficiency (판관비 효율)** : saleq / xsgaq
해자가 있는 기업은 광고비 등을 덜 쓰고도 잘 팔린다.

**NOPAT-ROA (세후 영업 자산수익률)** : oiadpq × (1 − 유효세율) / atq
세후 영업이익(NOPAT)을 총자산으로 나눈 자본 효율성 지표. 유효세율(txtq/piq)은 세전이익이
양수일 때만 취하고 [0, 0.5]로 클리핑하며, 그 외에는 법정세율 21%를 가정한다.

> 투하자본(총부채+자기자본) 구성 항목을 추출하지 않았으므로 엄밀한 ROIC 대신
> 총자산 기준 NOPAT-ROA를 사용한다. 두 지표는 방향성은 같지만 수준이 다르다.


In [ ]:
fund_data = db.raw_sql("""
            SELECT
                f.gvkey, f.datadate, f.tic AS ticker, f.cusip, f.conm,
                f.saleq, f.revtq, f.cogsq, f.xsgaq, f.oiadpq, f.txtq, f.piq, f.atq,

                (f.revtq - f.cogsq) / NULLIF(f.revtq, 0) AS gross_margin,
                (f.saleq / NULLIF(f.xsgaq, 0)) AS marketing_effi,
                -- NOPAT-ROA: 세후 영업이익 / 총자산 (유효세율은 piq>0일 때만, [0,0.5] 클리핑, 그 외 21%)
                f.oiadpq * (1 - CASE WHEN f.piq > 0
                                     THEN LEAST(GREATEST(f.txtq / f.piq, 0), 0.5)
                                     ELSE 0.21 END)
                / NULLIF(f.atq, 0) AS nopat_roa
            FROM comp.fundq AS f
            INNER JOIN
                comp.company AS c ON f.gvkey = c.gvkey
            WHERE
                c.gind = '453010'
                AND f.datadate >= '2010-01-01'
                AND f.indfmt = 'INDL'
                AND f.datafmt = 'STD'
                AND f.popsrc = 'D'
                AND f.consol = 'C'
            """)
fund_data

In [ ]:
# 일별 수정주가 (분기말 시점 매칭용)
# 한계: msenames의 ticker로 매칭하면 ticker 재사용 종목이 섞일 수 있다.
# 정석은 crsp.ccmxpf_linktable로 gvkey→permno를 잇는 것 (wrds/PER 노트북 참고).
price_data = db.raw_sql("""
            SELECT
                m.ticker, d.date,
                ABS(d.prc) / NULLIF(d.cfacpr, 0) AS adj_price
            FROM crsp.dsf AS d
            INNER JOIN crsp.msenames AS m
                ON d.permno = m.permno
            WHERE m.namedt <= d.date AND d.date <= m.nameendt
            AND d.date >= '2010-01-01'
            AND m.ticker IN (SELECT ticker FROM comp.company WHERE gind = '453010')
""")
price_data

In [ ]:
fund_data["datadate"] = pd.to_datetime(fund_data["datadate"])
price_data["date"] = pd.to_datetime(price_data["date"])

price_data = price_data.dropna(subset = ["adj_price"])

fund_data = fund_data.sort_values(by = "datadate")
price_data = price_data.sort_values(by = "date")

merged_data = pd.merge_asof(
    fund_data,
    price_data,
    left_on = 'datadate',
    right_on = 'date',
    by = 'ticker',
    direction = 'backward'
)

merged_data = merged_data.drop(columns = ["date"])
merged_data["rev_growth_yoy"] = merged_data.groupby('ticker')['revtq'].pct_change(periods = 4, fill_method = None)

import numpy as np
merged_data = merged_data.replace([np.inf, -np.inf], np.nan).dropna(subset = ["rev_growth_yoy", "gross_margin"])

merged_data

In [ ]:
merged_data.to_csv("Moat_Analysis.csv", index=False)